In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.optim import Adam
from torchvision import datasets
from torchvision import models
from torchvision.models import EfficientNet_V2_L_Weights
from torchvision.models import ViT_L_16_Weights
from torch.utils.data import DataLoader, random_split
import random

# ---------------- DRIVE PATH ----------------
DATASET_ROOT = "/content/drive/MyDrive/Chatbot/SoilClassificationDataset/CyAUG-Dataset"
SAVE_DIR = "/content/drive/MyDrive/Chatbot/soil_models"

os.makedirs(SAVE_DIR, exist_ok=True)

# ---------------- SEED ----------------
seed = 42
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# ---------------- IMAGE SIZE ----------------
img_size = (256,256)

mean = [0.485,0.456,0.406]
std = [0.229,0.224,0.255]

transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean,std)
])

# ---------------- DATASET ----------------
dataset = datasets.ImageFolder(DATASET_ROOT, transform=transform)

num_classes = len(dataset.classes)

# ---------------- SPLIT ----------------
total_size = len(dataset)

train_size = int(0.7 * total_size)
valid_size = int(0.15 * total_size)
test_size = total_size - train_size - valid_size

train_dataset, valid_dataset, test_dataset = random_split(
    dataset,
    [train_size, valid_size, test_size],
    generator=torch.Generator().manual_seed(seed)
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=4, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

# ---------------- ENSEMBLE MODEL ----------------
class Ensemble(nn.Module):

    def __init__(self,model1,model2,num_classes,f1,f2):

        super().__init__()

        self.model1 = model1
        self.model2 = model2

        self.model1.classifier[1] = nn.Identity()
        self.model2.heads[0] = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Linear(f1+f2,512),
            nn.ReLU(),
            nn.Linear(512,num_classes)
        )

        self.resize224 = transforms.Resize((224,224))

    def forward(self,x):

        x1 = self.model1(x)
        x1 = x1.view(x1.size(0),-1)

        x2 = torch.stack([self.resize224(img) for img in x])
        x2 = self.model2(x2)
        x2 = x2.view(x2.size(0),-1)

        x = torch.cat((x1,x2),dim=1)
        x = self.classifier(x)

        return x

# ---------------- MODEL 1 ----------------
model1 = models.efficientnet_v2_l(
    weights = EfficientNet_V2_L_Weights.IMAGENET1K_V1
)

f1 = model1.classifier[1].in_features
model1.classifier[1] = nn.Linear(f1,num_classes)

# ---------------- MODEL 2 ----------------
model2 = models.vit_l_16(
    weights = ViT_L_16_Weights.IMAGENET1K_V1
)

f2 = model2.heads[0].in_features
model2.heads[0] = nn.Linear(f2,num_classes)

# ---------------- ENSEMBLE ----------------
model = Ensemble(model1,model2,num_classes,f1,f2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(),lr=0.0001,weight_decay=1e-8)

# ---------------- TRAINING ----------------
epochs = 15

for epoch in range(epochs):

    model.train()

    running_loss = 0
    correct = 0

    for inputs,labels in train_loader:

        inputs,labels = inputs.to(device),labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _,pred = torch.max(outputs,1)

        correct += (pred==labels).sum().item()

    train_acc = 100 * correct / len(train_loader.dataset)

    # ---------------- VALIDATION ----------------
    model.eval()

    val_correct = 0

    with torch.no_grad():

        for inputs,labels in valid_loader:

            inputs,labels = inputs.to(device),labels.to(device)

            outputs = model(inputs)

            _,pred = torch.max(outputs,1)

            val_correct += (pred==labels).sum().item()

    val_acc = 100 * val_correct / len(valid_loader.dataset)

    print("Epoch:",epoch+1)
    print("Train Acc:",train_acc)
    print("Val Acc:",val_acc)

    # ---------------- SAVE MODEL ----------------
    save_path = os.path.join(
        SAVE_DIR,
        f"soil_model_epoch_{epoch+1}.pth"
    )

    torch.save(model.state_dict(),save_path)

print("Training finished")

# ---------------- FINAL TEST ----------------
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for inputs,labels in test_loader:

        inputs,labels = inputs.to(device),labels.to(device)

        outputs = model(inputs)

        _,pred = torch.max(outputs,1)

        total += labels.size(0)

        correct += (pred==labels).sum().item()

test_acc = 100 * correct / total

print("Test Accuracy:",test_acc)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_l-59c71312.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_l-59c71312.pth


100%|██████████| 455M/455M [00:01<00:00, 295MB/s]


Downloading: "https://download.pytorch.org/models/vit_l_16-852ce7e3.pth" to /root/.cache/torch/hub/checkpoints/vit_l_16-852ce7e3.pth


100%|██████████| 1.13G/1.13G [00:05<00:00, 222MB/s]


Epoch: 1
Train Acc: 89.29072049341183
Val Acc: 87.30366492146597
Epoch: 2
Train Acc: 94.11269974768713
Val Acc: 93.717277486911
Epoch: 3
Train Acc: 95.51443790299972
Val Acc: 93.58638743455498
Epoch: 4
Train Acc: 96.60779366414354
Val Acc: 89.65968586387434
Epoch: 5
Train Acc: 96.41155032239978
Val Acc: 92.01570680628272
Epoch: 6
Train Acc: 97.33669750490608
Val Acc: 93.97905759162303
Epoch: 7
Train Acc: 97.39276703111858
Val Acc: 92.14659685863874
Epoch: 8
Train Acc: 97.44883655733109
Val Acc: 96.07329842931937
Epoch: 9
Train Acc: 97.89739276703112
Val Acc: 93.06282722513089
Epoch: 10
Train Acc: 97.64507989907486
Val Acc: 93.45549738219896
Epoch: 11
Train Acc: 97.64507989907486
Val Acc: 94.50261780104712
Epoch: 12
Train Acc: 98.23380992430614
Val Acc: 95.41884816753927
Epoch: 13
Train Acc: 98.40201850294365
Val Acc: 95.68062827225131
Epoch: 14
Train Acc: 98.90664423885619
Val Acc: 86.64921465968587
Epoch: 15
Train Acc: 97.92542753013737
Val Acc: 95.0261780104712
Training finished
Test

In [ ]:
!ls /content/drive/MyDrive/Chatbot/SoilClassificationDataset

ls: cannot access '/content/drive/MyDrive/Chatbot/SoilClassificationDataset': No such file or directory
